In [ ]:
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
from typing import List
import torch
import torch.nn as nn


class Saw(nn.Module):
    def forward(self, x):
        return torch.abs(torch.remainder(x, 1.0) - 0.5) - 0.25


class WavSaw(nn.Module):
    def forward(self, x):
        s = 1 / (1 + torch.abs(x))
        return s * (torch.abs(torch.remainder(x, 1.0) - 0.5) - 0.25)


class Abs(nn.Module):
    def forward(self, x):
        return torch.abs(x)


class MLP(nn.Module):
    def __init__(self, seq: List[int | nn.Module]):
        super().__init__()

        layers = []
        dim = seq[0]

        for item in seq[1:]:
            if isinstance(item, int):
                assert isinstance(dim, int)
                layers.append(nn.Linear(dim, item))
                dim = item
            elif isinstance(item, nn.Module):
                layers.append(item)
            else:
                raise ValueError(f"Invalid item in sequence: {item}")

        self.net = nn.Sequential(*layers)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.5)
            nn.init.normal_(m.bias, 0.0, 0.5)

    def forward(self, x):
        return self.net(x)


class MountainNoise(nn.Module):
    def __init__(self):
        super().__init__()

        # self.components = nn.ModuleList([MLP([2, 16, WavSaw(), 1]) for _ in range(4)])
        # self.acomponents = nn.ModuleList(
        #     [MLP([2, 16, WavSaw(), 1, Abs()]) for _ in range(4)]
        # )

        self.components = nn.ModuleList([MLP([2, 32, Saw(), 1]) for _ in range(4)])
        self.acomponents = nn.ModuleList(
            [MLP([2, 32, Saw(), 1, Abs()]) for _ in range(4)]
        )

    def forward(self, x):
        # ys = -torch.abs(
        #     torch.stack(
        #         [
        #             (0.5**i) * component((2.0**i) * x)
        #             for i, component in enumerate(self.components)
        #         ],
        #         dim=0,
        #     )
        # )  # [n_components, batch, 1]
        # r = torch.min(ys, dim=0, keepdim=False).values

        ys = torch.abs(
            torch.stack(
                [
                    (0.5**i) * component((2.0**i) * x)
                    for i, component in enumerate(self.components)
                ],
                dim=0,
            )
        )  # [n_components, batch, 1]
        r = -torch.max(ys, dim=0, keepdim=False).values

        # r = sum(component(x) for component in self.components) / len(self.components)

        # r = sum(component(x) for component in self.acomponents) / len(self.acomponents)

        # r = sum(
        #     (0.5**i) * component((0.8**i) * x)
        #     for i, component in enumerate(self.components)
        # ) / len(self.components)

        # r = sum(
        #     (0.5**i) * component((0.8**i) * x)
        #     for i, component in enumerate(self.acomponents)
        # ) / len(self.acomponents)

        # r = -sum(
        #     (0.5**i) * component((0.8**i) * x)
        #     for i, component in enumerate(self.acomponents)
        # ) / len(self.acomponents)

        # r = sum(component(x) for component in self.acomponents) / len(self.acomponents)

        e = 1 - r

        return e / (1 + torch.norm(x, dim=-1, keepdim=True) ** 2)


model = MountainNoise()
model.eval()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def evaluate_on_grid(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x = np.linspace(x_range[0], x_range[1], resolution)
    y = np.linspace(y_range[0], y_range[1], resolution)
    xx, yy = np.meshgrid(x, y)
    grid = np.stack([xx.flatten(), yy.flatten()], axis=-1)

    with torch.no_grad():
        inputs = torch.from_numpy(grid).float()
        raw = model(inputs).detach().cpu().numpy()
        if raw.ndim == 2 and raw.shape[1] == 1:
            raw = raw[:, 0]
        elif raw.ndim != 1:
            raise ValueError(
                f"Model must return one value per grid point, got shape {raw.shape}"
            )
        outputs = raw.reshape(resolution, resolution)

    return x, y, xx, yy, outputs


def plot_2d_3d(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x, y, xx, yy, outputs = evaluate_on_grid(
        model, x_range=x_range, y_range=y_range, resolution=resolution
    )

    # 2D heatmap and 3D surface side-by-side
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "heatmap"}, {"type": "surface"}]],
        subplot_titles=("2D Heatmap", "3D Surface"),
        horizontal_spacing=0.08,
    )

    fig.add_trace(
        go.Heatmap(
            z=outputs, x=x, y=y, colorscale="Viridis", colorbar=dict(title="output")
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Surface(z=outputs, x=xx, y=yy, colorscale="Viridis", showscale=False),
        row=1,
        col=2,
    )

    fig.update_xaxes(title_text="x", row=1, col=1)
    fig.update_yaxes(title_text="y", row=1, col=1)
    fig.update_scenes(
        xaxis_title="x", yaxis_title="y", zaxis_title="output", row=1, col=2
    )
    fig.update_layout(
        height=520, width=1200, title=f"Output over {x_range} x {y_range}"
    )
    fig.show()


s = 2
r = (-s, s)
plot_2d_3d(model, x_range=r, y_range=r, resolution=100)